# Lab 5 — Test-Driven Prompting

**Module:** LLMs for Software Engineering | **Duration:** 90 min | **Switch roles at Part 2**

**Fil Rouge:** DevAssist — AI-Powered Developer Documentation Assistant for TaskFlow

---

## Core Principle

> **The model proposes, tests dispose.**
> "Probable" code is not "correct" code. Only external oracles — tests — verify semantics.

## TDP Workflow

```
Spec + Test signatures ──► Prompt model ──► Run tests ──► PASS? ──► Done
       ▲                                        │          │
       │                                        ▼          ▼ NO
       │                                   Analyze failures
       │                                        │
       └────────── Refine prompt ◄──────────────┘
```

---
## Setup

In [ ]:
import json, sys, os
sys.path.insert(0, '.')

from utils.generation_utils import generate, is_ollama_available
from utils.code_runner import run_tests, extract_python_code

OLLAMA_AVAILABLE = is_ollama_available()
print(f"Ollama: {'✓ available' if OLLAMA_AVAILABLE else '⚠ unavailable — use precomputed'}")

with open('data/precomputed_outputs.json') as f:
    PRECOMPUTED = json.load(f)
print('✓ All imports and data loaded.')

---
# PART 1 — TEST-DRIVEN PROMPTING: `merge_intervals` (40 min)

**🟢 Driver starts here.**

---

## Step 1: Write Tests FIRST (10 min)

**Before writing ANY prompt**, open `tests/test_intervals.py` and fill in all test cases.

Your tests define what "correct" means. Cover:
- Normal: overlapping intervals
- No overlap
- Touching intervals (e.g., [1,3] and [3,5])
- Subset intervals (e.g., [1,10] and [3,5])
- Single interval
- Empty input
- Unsorted input
- Duplicates, negatives

In [ ]:
# Verify your tests file exists and has content
with open('tests/test_intervals.py') as f:
    test_content = f.read()
    n_tests = test_content.count('def test_')
    n_pass = test_content.count('pass')
print(f'Test functions found: {n_tests}')
if n_pass > 3:
    print(f'⚠ {n_pass} test bodies are still `pass` — fill them in!')
else:
    print('✓ Tests appear to have assertions.')

---
## Step 2: Prompt the Model — Iteration V1 (10 min)

Write a TDP prompt: include the specification, type hints, and test signatures.

In [ ]:
# ============================================================
# ITERATION V1: Your first TDP prompt
# ============================================================
# Include:
# - Function signature with type hints
# - Clear specification (what merge means)
# - At least one example input → output
# - Edge case behavior (empty, touching intervals)
# - Format: return ONLY the function, no explanation
# ============================================================

tdp_prompt_v1 = """
TODO: Write your V1 prompt here.

Implement this Python function:

def merge_intervals(intervals: list[list[int]]) -> list[list[int]]:
    ...

Specification:
- Takes a list of [start, end] interval pairs
- Merges all overlapping intervals
- Returns merged intervals sorted by start
- Touching intervals (e.g., [1,3] and [3,5]) should merge
- Empty input returns empty list

Example: merge_intervals([[1,3],[2,6],[8,10],[15,18]]) → [[1,6],[8,10],[15,18]]

Return ONLY the Python function. No explanation, no tests.
"""

if OLLAMA_AVAILABLE:
    response_v1 = generate(tdp_prompt_v1)
    print("=== MODEL OUTPUT (V1) ===")
    print(response_v1)
else:
    print("Using precomputed V1 output:")
    response_v1 = PRECOMPUTED['merge_intervals_v1']['generated_code']
    print(response_v1)

In [ ]:
# === Extract and save the generated code ===
code_v1 = extract_python_code(response_v1)
print("Extracted code:")
print(code_v1)

## Step 3: Run Tests Against V1

In [ ]:
# Write V1 code to taskflow/intervals.py and run tests
with open('taskflow/intervals.py', 'w') as f:
    f.write('"""Auto-generated by Lab 5 — V1."""\n\n')
    f.write(code_v1 + '\n')

result_v1 = run_tests(code_v1, 'tests/test_intervals.py')
print(f"\n{'='*50}")
print(f"V1 Results: {result_v1['passed']} passed, {result_v1['failed']} failed, {result_v1['errors']} errors")
print(f"{'='*50}")
print(result_v1['stdout'])
if result_v1['stderr']:
    print("STDERR:", result_v1['stderr'][:500])

### V1 Analysis

**Which tests passed?** TODO

**Which tests failed? Why?** TODO

**Diagnosis:** What did the model get wrong? TODO

---
## Step 4: Iterate — V2 (10 min)

Refine your prompt based on V1 failures. Use mechanistic reasoning:
- Missing edge case? → Add it as a few-shot example (attention over patterns)
- Wrong algorithm? → Add explicit constraints ("sort first")
- Formatting issue? → Add negative constraints

In [ ]:
# ============================================================
# ITERATION V2: Refined prompt based on V1 failures
# ============================================================

tdp_prompt_v2 = """
TODO: Refine your prompt. What did you change and why?
"""

if OLLAMA_AVAILABLE:
    response_v2 = generate(tdp_prompt_v2)
    code_v2 = extract_python_code(response_v2)
    print("=== MODEL OUTPUT (V2) ===")
    print(code_v2)
else:
    print("Using precomputed V1 (correct) output for V2:")
    code_v2 = PRECOMPUTED['merge_intervals_v1']['generated_code']
    print(code_v2)

In [ ]:
# Run tests against V2
with open('taskflow/intervals.py', 'w') as f:
    f.write('"""Auto-generated by Lab 5 — V2."""\n\n')
    f.write(code_v2 + '\n')

result_v2 = run_tests(code_v2, 'tests/test_intervals.py')
print(f"\n{'='*50}")
print(f"V2 Results: {result_v2['passed']} passed, {result_v2['failed']} failed")
print(f"{'='*50}")
print(result_v2['stdout'])

### V2 Analysis

**What changed?** TODO

**Mechanistic reasoning for the change:** TODO

**Result improvement:** TODO

In [ ]:
# === V3 (if needed) ===

tdp_prompt_v3 = """
TODO: Only if V2 still has failures.
"""

# if OLLAMA_AVAILABLE:
#     response_v3 = generate(tdp_prompt_v3)
#     code_v3 = extract_python_code(response_v3)
#     with open('taskflow/intervals.py', 'w') as f:
#         f.write('"""V3."""\n\n' + code_v3 + '\n')
#     result_v3 = run_tests(code_v3, 'tests/test_intervals.py')
#     print(f"V3: {result_v3['passed']} passed, {result_v3['failed']} failed")
#     print(result_v3['stdout'])

### Iteration Summary — Part 1

| Iteration | Prompt Strategy | Pass Rate | Key Change |
|-----------|----------------|-----------|------------|
| V1 | TODO | TODO/10 | Initial |
| V2 | TODO | TODO/10 | TODO |
| V3 | TODO | TODO/10 | TODO |

**Most impactful prompt change:** TODO

**Copy this table into `iteration_log.md` with full prompts.**

---
# PART 2 — AI TEST GENERATION + REFACTORING (25 min)

**🔄 Switch driver/navigator.**

---

## Exercise A: AI-Generated Tests for `find_second_largest` (15 min)

The function in `taskflow/stats.py` is correct but uses O(n log n) sorting.
First, ask the model to generate tests. Then critically evaluate them.

In [ ]:
# Read the function to include in prompt
with open('taskflow/stats.py') as f:
    stats_code = f.read()
print(stats_code)

In [ ]:
# ============================================================
# EXERCISE A: Prompt the model to generate pytest tests
# ============================================================

test_gen_prompt = f"""Generate a comprehensive pytest test suite for this Python function.

```python
{stats_code}
```

Requirements:
- Use pytest (import pytest)
- Import: from taskflow.stats import find_second_largest
- Include edge cases: empty list, single element, all identical, negative numbers
- Test the ValueError for fewer than 2 unique values
- Each test should have a descriptive name
- Return ONLY the test code, nothing else
"""

if OLLAMA_AVAILABLE:
    test_response = generate(test_gen_prompt, max_tokens=1000)
    print("=== AI-GENERATED TESTS ===")
    print(test_response)
else:
    print("Using precomputed AI-generated tests:")
    test_response = PRECOMPUTED['find_second_largest_tests']['ai_generated_tests']
    print(test_response)

In [ ]:
# Extract the test code
generated_tests = extract_python_code(test_response)

# Save to test file for review
with open('tests/test_stats.py', 'w') as f:
    f.write('"""AI-generated tests for find_second_largest — Lab 5, Part 2."""\n\n')
    f.write(generated_tests + '\n')

print('✓ AI-generated tests saved to tests/test_stats.py')
print(f'Test functions: {generated_tests.count("def test_")}')

In [ ]:
# Run the AI-generated tests
result_tests = run_tests('', 'tests/test_stats.py')
print(f"AI-generated test results: {result_tests['passed']} passed, {result_tests['failed']} failed")
print(result_tests['stdout'])

### Critical Evaluation of AI-Generated Tests

**Q1: How many tests did the model generate?** TODO

**Q2: Edge cases covered?**
- [ ] Empty list
- [ ] Single element
- [ ] All identical values
- [ ] Negative numbers
- [ ] Mixed positive/negative
- [ ] Very large list
- [ ] Two elements
- [ ] ValueError for < 2 unique values

**Q3: Any WRONG tests (incorrect expected values)?** TODO

**Q4: Any TRIVIAL tests (don't test anything meaningful)?** TODO

**Q5: Would you ship these tests as-is?** TODO

**Q6: What edge cases did the model miss?** TODO

---
## Exercise B: Guided Refactoring (10 min)

Refactor `find_second_largest` from O(n log n) to O(n) single-pass.
The AI-generated tests (plus your additions) serve as the safety net.

In [ ]:
# ============================================================
# EXERCISE B: Guided Refactoring
# ============================================================

refactor_prompt = f"""Refactor the following Python function to:
1. Use a single-pass O(n) algorithm instead of sorting
2. Handle edge cases identically (raise ValueError for < 2 unique values)
3. Preserve the same function signature and docstring

Current implementation:
```python
{stats_code}
```

Return ONLY the refactored function. No explanation.
"""

if OLLAMA_AVAILABLE:
    refactored_response = generate(refactor_prompt, max_tokens=600)
    refactored_code = extract_python_code(refactored_response)
else:
    refactored_code = PRECOMPUTED['refactored_find_second_largest']['code']

print("=== REFACTORED CODE ===")
print(refactored_code)

In [ ]:
# Run existing tests against refactored code
with open('taskflow/stats.py', 'w') as f:
    f.write('"""Refactored by AI — Lab 5, Part 2."""\n\n')
    f.write(refactored_code + '\n')

result_refactor = run_tests('', 'tests/test_stats.py')
print(f"Refactored code: {result_refactor['passed']} passed, {result_refactor['failed']} failed")
print(result_refactor['stdout'])

### Refactoring Evaluation

**Q1: Does the refactored code pass all tests?** TODO

**Q2: Is the algorithm actually O(n)?** (or did the model sneak in sorting?) TODO

**Q3: Is the code more or less readable?** TODO

**Q4: Did the model introduce any new bugs?** TODO

**Q5: Mechanistic explanation — why did including the current implementation help?** (Hint: it's a few-shot example of the behavior + "O(n)" constrains the algorithm choice)

TODO

---
# PART 3 — REFACTOR `format_task_summary` (10 min)

The function in `taskflow/formatting.py` works but is messy. Comprehensive tests
are already provided in `tests/test_formatting.py` (13 test cases).

Your job: prompt the model to refactor, then verify ALL tests pass.

In [ ]:
# Read the current messy implementation
with open('taskflow/formatting.py') as f:
    formatting_code = f.read()
print(formatting_code)

In [ ]:
# Verify all 13 tests pass BEFORE refactoring
result_pre = run_tests('', 'tests/test_formatting.py')
print(f"Pre-refactoring: {result_pre['passed']} passed, {result_pre['failed']} failed")
print(result_pre['stdout'])

In [ ]:
# ============================================================
# REFACTORING PROMPT
# ============================================================

refactor_format_prompt = f"""Refactor this Python function to be cleaner and more Pythonic:
- Use dict.get() with defaults instead of if/else chains
- Build lines in a list and join at the end
- Preserve EXACTLY the same output format and behavior
- Keep the same function signature and docstring

Current implementation:
```python
{formatting_code}
```

Return ONLY the refactored function. No explanation.
"""

if OLLAMA_AVAILABLE:
    refactored_fmt = extract_python_code(generate(refactor_format_prompt, max_tokens=600))
else:
    refactored_fmt = PRECOMPUTED['refactored_format_task_summary']['code']

print("=== REFACTORED format_task_summary ===")
print(refactored_fmt)

In [ ]:
# Save and run existing tests against refactored code
with open('taskflow/formatting.py', 'w') as f:
    f.write('"""Refactored by AI — Lab 5, Part 3."""\n\n')
    f.write(refactored_fmt + '\n')

result_post = run_tests('', 'tests/test_formatting.py')
print(f"\n{'='*50}")
print(f"Post-refactoring: {result_post['passed']} passed, {result_post['failed']} failed")
print(f"{'='*50}")
print(result_post['stdout'])

if result_post['all_passed']:
    print('\n✓ Refactoring preserved behavior — all tests pass!')
else:
    print('\n✗ Refactoring BROKE something — check failures above.')

### Part 3 Observation

**Did tests pass after refactoring?** TODO

**Is the refactored code better?** TODO

**If tests failed — what broke? Why?** TODO

---
# WRAP-UP

## Key Takeaways

1. TODO
2. TODO
3. TODO

## Checklist

- ✅ `tests/test_intervals.py` — 10+ test cases filled in
- ✅ `taskflow/intervals.py` — working AI-generated `merge_intervals`
- ✅ `tests/test_stats.py` — AI-generated tests + your additions
- ✅ `taskflow/formatting.py` — refactored, all 13 tests pass
- ✅ `iteration_log.md` — prompts V1/V2/V3 + results + analysis
- ✅ `review.md` — AI code review (what it did well/poorly)
- ✅ `git add -A && git commit -m 'Lab 5 complete' && git push`

## Portfolio Entry

**Use Case #5:** Test-driven code generation — spec → prompt → test → iterate.
Include iteration log and test pass/fail progression.

## Preview: Lab 6

In **Lab 6 — RAG Pipeline**, your TaskFlow code + tests from today become
part of the knowledge base that DevAssist retrieves from when answering questions.

---
*Lab 5 of 8 — DevAssist / TaskFlow Lab Series*